Copias impresas y electrónicas de *Modelado y simulación en Python* están disponibles en [No Starch Press](https://nostarch.com/modeling-and-simulation-python) y [Bookshop.org](https://bookshop.org/p/books/modeling-and-simulation-in-python-allen-b-downey/17836697?ean=9781718502161) y [Amazon](https://amzn.to/3y9UxNb).

# Café refrescante

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://github.com/AllenDowney/ModSimPy/raw/master/' +
         'modsim.py')

In [2]:
# import functions from modsim

from modsim import *

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Haga clic aquí para acceder a los cuadernos: <https://allendowney.github.io/ModSimPy/>.

Hasta ahora los sistemas que hemos estudiado han sido físicos en el sentido de que existen en el mundo, pero no han sido físicos en el sentido en que suelen tratarse las clases de física. En los próximos capítulos, abordaremos algo de física, comenzando con *sistemas térmicos*, es decir, sistemas donde la temperatura de los objetos cambia a medida que el calor se transfiere de uno a otro.

## El problema del enfriamiento del café

El problema del enfriamiento del café fue discutido por Jearl Walker en 
"The Amateur Scientist", *Scientific American*, Volumen 237, Número 5, noviembre de 1977. Desde entonces se ha convertido en un ejemplo estándar de modelado y simulación.

Aquí está mi versión del problema:

> Supongamos que me detengo camino al trabajo para tomar una taza de café y un pequeño recipiente de leche. Suponiendo que quiero que el café esté lo más caliente posible cuando llegue al trabajo, ¿debo agregar la leche en la cafetería, esperar hasta llegar al trabajo o agregar la leche en algún momento intermedio?

Para ayudar a responder esta pregunta, hice una prueba con la leche y
café en recipientes separados y tomé algunas medidas (en realidad no):

- Al momento de servir, la temperatura del café es de 90 °C. El volumen es
    300 ml.

- La leche está a una temperatura inicial de 5 °C, y tardo unos
    50 ml.

- La temperatura ambiente en mi coche es de 22 °C.

- El café se sirve en una taza bien aislada. Cuando llego al trabajo después de 30 minutos, la temperatura del café ha bajado a 70 °C.

- El recipiente de la leche no está bien aislado. Después de 15 minutos,
    Se calienta hasta 20 °C, casi la temperatura ambiente.

Para utilizar estos datos y responder la pregunta, tenemos que saber algo.
sobre temperatura y calor, y tenemos que tomar algunas decisiones de modelado.

## Temperatura y calor

Para entender cómo se enfría el café (y se calienta la leche), necesitamos un modelo de
temperatura y calor. *Temperatura* es una propiedad de un objeto o un
sistema; en unidades SI se mide en grados Celsius (°C). La temperatura cuantifica qué tan caliente o frío está un objeto, lo cual está relacionado con la velocidad promedio de las partículas que lo componen.

Cuando las partículas de un objeto caliente entran en contacto con las de un objeto frío, la
El objeto caliente se enfría y el objeto frío se calienta a medida que se consume energía.
transferidos de uno a otro. La energía transferida se llama
*calor*; en unidades SI se mide en julios (J).

El calor está relacionado con la temperatura mediante la siguiente ecuación (ver
<http://modsimpy.com/thermass>): 

$$Q = C~\Delta T$$ 

donde $Q$ es la cantidad de calor transferido a un objeto, $\Delta T$ es el cambio de temperatura resultante y $C$ es la *masa térmica* del objeto, que es una propiedad del objeto que determina cuánta energía se necesita para calentarlo o enfriarlo. En unidades SI, la masa térmica se mide en julios por grado Celsius (J/°C).

Para objetos hechos principalmente de un material, la masa térmica puede ser
calculado así: 

$$C = m c_p$$ 

donde $m$ es la masa del objeto y $c_p$ es la *capacidad calorífica específica* del material, que es la cantidad de masa térmica por gramo (ver <http://modsimpy.com/specheat>).

Podemos usar estas ecuaciones para estimar la masa térmica de una taza de
café. La capacidad calorífica específica del café probablemente se acerque a esa
de agua, que es 4,2 J/g/°C. Suponiendo que la densidad del café es
cercana a la del agua, que es 1 g/mL, la masa de 300 mL de café es 300 g y la masa térmica es 1260 J/°C.

Entonces, cuando una taza de café se enfría de 90 °C a 70 °C, el cambio en
temperatura, $\Delta T$ es 20 °C, lo que significa que 25 200 J de calor
La energía se transfirió desde la taza y el café al entorno circundante.
(el portavasos y el aire de mi auto).

Para darle una idea de cuánta energía es, si pudiera
aprovechar todo ese calor para hacer trabajo (que no puedes), podrías
Úselo para elevar una taza de café desde el nivel del mar hasta 8571 m, apenas por debajo de la altura del Monte Everest, 8848 m.

## Transferencia de calor

En una situación como el problema del enfriamiento del café, hay tres maneras
transferencias de calor de un objeto a otro (ver <http://modsimpy.com/transfer>):

- Conducción: Cuando entran en contacto objetos a diferentes temperaturas.
    contacto, las partículas que se mueven más rápido de la temperatura más alta
    El objeto transfiere energía cinética a las partículas que se mueven más lentamente del objeto de menor temperatura.

- Convección: Cuando las partículas de un gas o líquido fluyen de un lugar a otro.
    lugar, llevan consigo energía térmica. Los flujos de fluidos pueden ser causados
    por acción externa, como agitación, o por diferencias internas en
    temperatura. Por ejemplo, es posible que hayas oído que el aire caliente sube,
    que es una forma de "convección natural".

- Radiación: cuando las partículas de un objeto se mueven debido a la energía térmica,
    Emiten radiación electromagnética. La energía transportada por este
    La radiación depende de la temperatura del objeto y de las propiedades de la superficie.
    (ver <http://modsimpy.com/thermrad>).

Para objetos como el café en un automóvil, el efecto de la radiación es mucho
más pequeño que los efectos de la conducción y la convección, por lo que lo ignoraremos.

La convección puede ser un tema complejo, ya que a menudo depende de detalles del flujo de fluidos en tres dimensiones. Pero para este problema podremos salirnos con la nuestra con un modelo simple llamado "ley de enfriamiento de Newton".

## Ley de enfriamiento de Newton

*La ley de enfriamiento de Newton* afirma que la tasa de cambio de temperatura de un objeto es proporcional a la diferencia de temperatura entre el objeto y el entorno que lo rodea:

$$\frac{dT}{dt} = -r (T - T_{env})$$ 

donde $t$ es el tiempo, $T$ es la temperatura del objeto, $T_{env}$ es la temperatura del ambiente y $r$ es una constante que caracteriza la rapidez con la que se transfiere el calor entre el objeto y el ambiente.

La llamada "ley" de Newton es en realidad un modelo: es una buena aproximación en algunas condiciones y menos buena en otras.

Por ejemplo, si el mecanismo principal de transferencia de calor es la conducción,
La ley de Newton es "verdadera", es decir, que $r$ es constante en un
amplio rango de temperaturas. Y a veces podemos estimar $r$ en función de
las propiedades materiales y la forma del objeto.

Cuando la convección contribuye con una fracción no despreciable de la transferencia de calor, $r$ depende de la temperatura, pero la ley de Newton suele ser bastante precisa, al menos en un rango estrecho de temperaturas. En este caso, $r$ normalmente debe estimarse experimentalmente, ya que depende de detalles de la forma de la superficie, el flujo de aire, la evaporación, etc.

Cuando la radiación constituye una parte sustancial de la transferencia de calor, la teoría de Newton
La ley no es un buen modelo en absoluto. Este es el caso de los objetos en el espacio o en el vacío, y de los objetos a altas temperaturas (más de unos pocos
cien grados Celsius, digamos).

Sin embargo, para una situación como el problema del enfriamiento del café, esperamos que el modelo de Newton sea bastante bueno.

Con eso, solo tenemos que tomar una decisión de modelado más: si tratamos el café y la taza como objetos separados o como un solo objeto. Si la taza es de papel, tiene menos masa que el café y la capacidad calorífica específica del papel también es menor. En ese caso, sería razonable tratar la taza y el café como un solo objeto. Para una taza con una masa térmica sustancial, como una taza de cerámica, podríamos considerar un modelo que calcule la temperatura del café y de la taza por separado.

## Implementación del enfriamiento newtoniano

Para empezar, nos centraremos en el café. Luego, como ejercicio, puedes simular la leche. En el próximo capítulo, los juntaremos, literalmente.

Aquí hay una función que toma los parámetros del sistema y crea un objeto `System`:

In [3]:
def make_system(T_init, volume, r, t_end):
    return System(T_init=T_init,
                  T_final=T_init,
                  volume=volume,
                  r=r,
                  t_end=t_end,
                  T_env=22,
                  t_0=0,
                  dt=1)

Además de los parámetros, `make_system` establece la temperatura del ambiente, `T_env`, la marca de tiempo inicial, `t_0`, y el paso de tiempo, `dt`, que usaremos para simular el proceso de enfriamiento.
Aquí hay un objeto `System` que representa el café.

In [4]:
coffee = make_system(T_init=90, volume=300, r=0.01, t_end=30)

Los valores de `T_init`, `volume` y `t_end` provienen del planteamiento del problema.
Elegí el valor de `r` de forma arbitraria por ahora; Veremos cómo estimarlo pronto.

Estrictamente hablando, la ley de Newton es una ecuación diferencial, pero en un corto período de tiempo podemos aproximarla con una ecuación en diferencias:

$$\Delta T = -r (T - T_{env}) dt$$ 

donde $dt$ es el paso de tiempo y $\Delta T$ es el cambio de temperatura durante ese paso de tiempo.

Nota: Utilizo $\Delta T$ para indicar un cambio de temperatura a lo largo del tiempo, pero en el contexto de la transferencia de calor, es posible que también vea $\Delta T$ utilizado para indicar la diferencia de temperatura entre un objeto y su
entorno, $T - T_{env}$. Para minimizar la confusión, evito este segundo
uso.

La siguiente función toma la hora actual `t`, la temperatura actual, `T` y un objeto `System`, y calcula el cambio de temperatura durante un paso de tiempo:

In [5]:
def change_func(t, T, system):
    r, T_env, dt = system.r, system.T_env, system.dt    
    return -r * (T - T_env) * dt

Podemos probarlo con la temperatura inicial del café, así:

In [6]:
change_func(0, coffee.T_init, coffee)

Con `dt=1` minuto, la temperatura desciende aproximadamente 0,7 °C, al menos para este valor de `r`.

Ahora aquí hay una versión de `run_simulation` que simula una serie de pasos de tiempo desde `t_0` hasta `t_end`:

In [7]:
def run_simulation(system, change_func):
    t_array = linrange(system.t_0, system.t_end, system.dt)
    n = len(t_array)
    
    series = TimeSeries(index=t_array)
    series.iloc[0] = system.T_init
    
    for i in range(n-1):
        t = t_array[i]
        T = series.iloc[i]
        series.iloc[i+1] = T + change_func(t, T, system)
    
    system.T_final = series.iloc[-1]
    return series

Hay dos cosas aquí que son diferentes de las versiones anteriores de `run_simulation`.

Primero, usamos `linrange` para crear una matriz de valores desde `t_0` hasta `t_end` con el paso de tiempo `dt`.
`linrange` es similar a `linspace`; ambos toman un valor inicial y un valor final y devuelven una matriz de valores igualmente espaciados.
La diferencia es el tercer argumento: `linspace` toma un número entero que indica la cantidad de puntos en el rango; `linrange` toma un tamaño de paso que indica el intervalo entre valores.
Cuando creamos `TimeSeries`, usamos el argumento de palabra clave `index` para indicar que el índice de `TimeSeries` es la matriz de marcas de tiempo, `t_array`.

En segundo lugar, esta versión de `run_simulation` usa `iloc` en lugar de `loc` para especificar las filas en `TimeSeries`.
Aquí está la diferencia: 

* Con `loc`, la etiqueta entre paréntesis puede ser cualquier tipo de valor, con cualquier inicio, fin y paso de tiempo.  Por ejemplo, en el modelo de población mundial, las etiquetas son años que comienzan en 1960 y terminan en 2016.

* Con `iloc`, la etiqueta entre paréntesis siempre es un número entero que comienza en 0. Por lo tanto, siempre podemos obtener el primer elemento con `iloc[0]` y el último elemento con `iloc[-1]`, independientemente de cuáles sean las etiquetas.

En esta versión de `run_simulation`, la variable de bucle es un número entero, `i`, que va de `0` a `n-1`, incluido `0` pero sin incluir `n-1`.
Entonces, la primera vez que realizamos el ciclo, `i` es `0` y el valor que agregamos a `TimeSeries` tiene el índice 1.
La última vez que realizamos el ciclo, `i` es `n-2` y el valor que agregamos tiene el índice `n-1`.

Podemos ejecutar la simulación así:

In [8]:
results = run_simulation(coffee, change_func)

El resultado es un `TimeSeries` con una fila por paso de tiempo. 
Aquí están las primeras filas:

In [9]:
show(results.head())

Y las últimas filas:

In [10]:
show(results.tail())

Con `t_0=0`, `t_end=30` y `dt=1`, las marcas de tiempo van de `0.0` a `30.0`.

Así es como se ve el `TimeSeries`.

In [11]:
results.plot(label='coffee')

decorate(xlabel='Time (min)',
         ylabel='Temperature (C)',
         title='Coffee Cooling')

La temperatura después de 30 minutos es de 72,3 °C, que es un poco más alta que la medición que intentamos igualar, que es de 70 °C.

In [12]:
coffee.T_final

Para encontrar el valor de `r` donde la temperatura final es exactamente 70 °C, podríamos proceder mediante prueba y error, pero es más eficiente utilizar un algoritmo de búsqueda de raíces.

## Encontrar raíces

La biblioteca ModSim proporciona una función llamada `root_scalar` que encuentra las raíces de ecuaciones no lineales. Como ejemplo, supongamos que desea encontrar las raíces del polinomio. 

$$f(x) = (x - 1)(x - 2)(x - 3)$$ 

Una *raíz* es un valor de $x$ que genera $f(x)=0$. Por la forma en que escribí este polinomio, podemos ver que si $x=1$, el primer factor es 0; si $x=2$, el segundo factor es 0; y si $x=3$, el tercer factor es 0, entonces esas son las raíces.

Usaré este ejemplo para demostrar `root_scalar`. Primero, tenemos que
escriba una función que evalúe $f$:

In [13]:
def func(x):
    return (x-1) * (x-2) * (x-3)

Ahora llamamos a `root_scalar` así:

In [14]:
res = root_scalar(func, bracket=[1.5, 2.5])
res

El primer argumento es la función cuyas raíces queremos. el segundo
El argumento es un intervalo que contiene o *entre corchetes* una raíz. El resultado es un objeto que contiene varias variables, incluido el valor booleano `converged`, que es `True` si la búsqueda convergió exitosamente en una raíz, y `root`, que es la raíz que se encontró.

In [15]:
res.root

Si proporcionamos un intervalo diferente, encontramos una raíz diferente.

In [16]:
res = root_scalar(func, bracket=[2.5, 3.5])
res.root

Si el intervalo no contiene una raíz, obtendrá un `ValueError` y un mensaje como `f(a) and f(b) must have different signs`.

Ahora podemos usar `root_scalar` para estimar `r`.

## Estimando r

Lo que queremos es el valor de `r` que produzca una temperatura final de
70°C. Para usar `root_scalar`, necesitamos una función que tome `r` como parámetro y devuelva la diferencia entre la temperatura final y la objetivo:

In [17]:
def error_func(r, system):
    system.r = r
    results = run_simulation(system, change_func)
    return system.T_final - 70

Esto se llama *función de error* porque devuelve el
diferencia entre lo que obtuvimos y lo que queríamos, es decir, el error.
Con el valor correcto de `r`, el error es 0.

Podemos probar `error_func` así, usando la suposición inicial `r=0.01`:

In [18]:
coffee = make_system(T_init=90, volume=300, r=0.01, t_end=30)
error_func(0.01, coffee)

El resultado es un error de 2,3 °C, lo que significa que la temperatura final con `r=0.01` es demasiado alta.

In [19]:
error_func(0.02, coffee)

Con `r=0.02`, el error es de aproximadamente -11 °C, lo que significa que la temperatura final es demasiado baja. Entonces sabemos que el valor correcto debe estar en el medio.

Ahora podemos llamar a `root_scalar` así:

In [20]:
res = root_scalar(error_func, coffee, bracket=[0.01, 0.02])
res

El primer argumento es la función de error.
El segundo argumento es el objeto `System`, que `root_scalar` pasa como argumento a `error_func`.
El tercer argumento es un intervalo que abarca la raíz.

Aquí está la raíz que encontramos.

In [21]:
r_coffee = res.root
r_coffee

En este ejemplo, `r_coffee` resulta ser aproximadamente `0.0115`, en unidades de min$^{-1}$ (minutos inversos).
Podemos confirmar que este valor es correcto configurando `r` en la raíz que encontramos y ejecutando la simulación.

In [22]:
coffee.r = res.root
run_simulation(coffee, change_func)
coffee.T_final

La temperatura final es muy cercana a los 70 °C.

## Resumen

Este capítulo presenta los conceptos básicos del calor, la temperatura y la ley de enfriamiento de Newton, que es un modelo preciso cuando la mayor parte de la transferencia de calor se realiza por conducción y convección, no por radiación.

Para simular una taza de café caliente, escribimos la ley de Newton como una ecuación en diferencias y luego escribimos una versión de `run_simulation` que la implementa. Luego usamos `root_scalar` para encontrar el valor de `r` que coincide con la medida de mi experimento hipotético.

Todo eso es el primer paso hacia la solución del problema del enfriamiento del café que planteé al principio del capítulo. Como ejercicio, realizarás el siguiente paso que es simular la leche. En el próximo capítulo, modelaremos el proceso de mezcla y resolveremos el problema.

## Ejercicios

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Puede acceder a los cuadernos en <https://allendowney.github.io/ModSimPy/>.

### Ejercicio 1

Simular la temperatura de 50 mL de leche con una temperatura inicial de 5 °C, en un recipiente con `r=0.1`, durante 15 minutos, y trazar los resultados. Utilice `make_system` para crear un objeto `System` que represente la leche y utilice `run_simulation` para simularlo.
Por prueba y error, encuentre un valor para `r` que acerque la temperatura final a 20 °C.

In [23]:
# Solution goes here

In [24]:
# Solution goes here

### Ejercicio 2

Escribe una función de error que simule la temperatura de la leche y devuelva la diferencia entre la temperatura final y 20 °C.  Úselo para estimar el valor de `r` para la leche.

In [25]:
# Solution goes here

In [26]:
# Solution goes here

In [27]:
# Solution goes here